# MISP API — Live Jupyter Demo
### Kraven Security · MISP Series

---
> **Run order matters**: Section 3 creates an event whose `id` is reused in later cells.
A cleanup cell at the bottom deletes the demo event so the instance resets between takes.

## ⚙️ Config & dependencies

API key is read from the `MISP_KEY` environment variable — **never hard-code a key in a notebook you're streaming.**
Set it before launching Jupyter:

```bash
export MISP_KEY="your-throwaway-key"      # macOS / Linux
# setx MISP_KEY "your-throwaway-key"      # Windows (new terminal after)
```

In [ ]:
# One-time install (run off-air). Uncomment if needed.
# %pip install pymisp pandas requests

In [ ]:
import os

MISP_URL    = "https://localhost"          # your throwaway instance
MISP_KEY    = os.environ["MISP_KEY"]       # loaded from env, not shown on screen
MISP_VERIFY = False                        # self-signed cert on the lab VM

# Quietly created later by Section 3; declared here so cleanup never NameErrors.
created = None

---
## 1. Setup & First Contact
*PyMISP, auth, and the object model*

PyMISP is the official Python wrapper around MISP's REST API. Everything it does, it does
by calling REST under the hood — we prove that in the appendix.

In [7]:
from pymisp import PyMISP
import urllib3
urllib3.disable_warnings()   # throwaway instance uses a self-signed cert

misp = PyMISP(MISP_URL, MISP_KEY, MISP_VERIFY)
misp.misp_instance_version   # connection sanity check

{'version': '2.5.36',
 'pymisp_recommended_version': '2.5.33.1',
 'perm_sync': True,
 'perm_sighting': True,
 'perm_galaxy_editor': True,
 'perm_analyst_data': True,
 'uuid': 'de8043d8-fe67-4c2d-b432-d113c22dd9a1',
 'request_encoding': ['gzip', 'br', 'zstd'],
 'filter_sightings': True}

**The object model bridge:** an *Event* in the UI is a `MISPEvent` here; an *Attribute*
is `add_attribute()`; an *Object* is a `MISPObject`. Same data model you already know — just code.

*Pitfalls:* `403` = wrong/expired key · SSL errors = the `verify=False` flag ·
if `misp_instance_version` misbehaves, fall back to `misp.get_version()`.

---
## 2. Search & Read
*Querying events, attributes, and IOCs — the workhorse*

In [8]:
# The 80% use case: recent network IOCs flagged for detection (to_ids=True)
results = misp.search(
    controller='attributes',
    type_attribute=['ip-src', 'ip-dst', 'domain', 'url'],
    to_ids=True,
    date_from='30d',          # MISP understands relative time
    pythonify=True            # return MISPAttribute objects, not raw JSON
)

print(f"{len(results)} attributes matched\n")
for attr in results[:10]:
    print(f"{attr.type:10} {attr.value:35.35} <- {attr.Event.info}")

61610 attributes matched

domain     world-new-iope.cc                   <- Maltrail IOC for 2026-04-09
domain     volimor.com                         <- Maltrail IOC for 2026-04-09
domain     app.rustture.cc                     <- Maltrail IOC for 2026-04-09
domain     gatuso.duckdns.org                  <- Maltrail IOC for 2026-04-09
domain     vmail.wiki                          <- Maltrail IOC for 2026-04-09
domain     shootr.cyou                         <- Maltrail IOC for 2026-04-09
domain     noto.space                          <- Maltrail IOC for 2026-04-09
domain     notospace.com                       <- Maltrail IOC for 2026-04-09
domain     docs.noto.space                     <- Maltrail IOC for 2026-04-09
ip-dst     91.208.197.241                      <- Maltrail IOC for 2026-04-09


**Two big levers:** `controller='attributes'` (IOC harvesting) vs `'events'` (campaign context),
and `pythonify=True` (Pythonic objects) vs default JSON (raw dict, closer to the REST response).

In [10]:
import pprint
# Same query, raw JSON — closer to what the REST API actually returns
raw = misp.search(controller='attributes', type_attribute='ip-dst',
                  to_ids=True, limit=3)
# print(raw[0]) # if raw else "no results — fetch the CIRCL OSINT feed first"

pprint.pprint(raw)


{'Attribute': [{'Event': {'Org': {'id': '1',
                                  'name': 'ADMIN',
                                  'uuid': 'f8af13d9-6302-49d2-9b58-e05f449ea0ec'},
                          'Orgc': {'id': '1',
                                   'name': 'ADMIN',
                                   'uuid': 'f8af13d9-6302-49d2-9b58-e05f449ea0ec'},
                          'ThreatLevel': {'id': '3', 'name': 'Low'},
                          'analysis': '0',
                          'date': '2026-04-16',
                          'distribution': '4',
                          'id': '1',
                          'info': 'test event 1',
                          'org_id': '1',
                          'orgc_id': '1',
                          'publish_timestamp': '1779457021',
                          'threat_level_id': '2',
                          'timestamp': '1779456937',
                          'user_id': '1',
                          'uuid': '279e93a7-f315-46e5-9c

In [11]:
# Analysts love this: events into a pandas DataFrame
import pandas as pd

events = misp.search(controller='events', tags=['tlp:white'], limit=20)   # JSON
df = pd.json_normalize([e['Event'] for e in events])
df[['id', 'info', 'date', 'threat_level_id']] if not df.empty else df

,id,info,date,threat_level_id
0,22,[CERT-FR] Infrastructure d'attaque du groupe c...,2021-02-08,2


---
## 3. Write
*Turn a threat report's IOC list into a structured MISP event — live*

After running the next cell, **flip to the MISP UI** and show the event appearing.

In [ ]:
from pymisp import MISPEvent

event = MISPEvent()
event.info          = "DEMO: IOCs from <public report name>"
event.distribution  = 0    # 0 = your org only
event.threat_level_id = 2  # 1 High / 2 Medium / 3 Low / 4 Undefined
event.analysis      = 1    # 0 Initial / 1 Ongoing / 2 Complete

# IOCs lifted straight from the report
event.add_attribute('domain', 'malicious-example.com', to_ids=True)
event.add_attribute('md5',    '44d88612fea8a8f36de82e1278abb02f', to_ids=True)
event.add_attribute('ip-dst', '203.0.113.10', to_ids=True)
event.add_tag('tlp:amber')

created = misp.add_event(event, pythonify=True)
print("Created event:", created.id, created.uuid)

Richer context: group related indicators with an **Object**, and attribute the activity
with a **Galaxy** cluster (threat actor).

In [ ]:
from pymisp import MISPObject

file_obj = MISPObject('file')
file_obj.add_attribute('filename', value='invoice_2026.exe')
file_obj.add_attribute('sha256',   value='275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f')
misp.add_object(created.id, file_obj)

# Attribute the activity with a Galaxy threat-actor cluster
misp.tag(created, 'misp-galaxy:threat-actor="Sofacy"')
print("Object + galaxy tag added to event", created.id)

---
## 4. Export & Integrate
*From MISP straight to your detection stack*

`return_format` is the killer feature — MISP renders the rules server-side, you just pick the target.

In [12]:
# IOCs as Suricata rules in a single call
suricata = misp.search(
    controller='attributes',
    tags=['tlp:white'],
    to_ids=True,
    return_format='suricata'
)
print(suricata[:1500])

# MISP export of IDS rules - optimized for suricata
#
# These NIDS rules contain some variables that need to exist in your configuration.
# Make sure you have set:
#
# $HOME_NET     - Your internal network range
# $EXTERNAL_NET - The network considered as outside
# $SMTP_SERVERS - All your internal SMTP servers
# $HTTP_PORTS   - The ports used to contain HTTP traffic (not required with suricata export)
# 
alert ip $HOME_NET any -> 135.181.97.81 any (msg: "MISP e22 Outgoing To IP: 135.181.97.81"; flow:not_established,to_server; flowbits:isnotset,outbound_ioc; flowbits:set,outbound_ioc;  classtype:bad-unknown; sid:7346785; rev:1; priority:2; reference:url,https://docker01/events/view/22; metadata:misp_event_uuid 6021536f-a808-4b9c-8136-d7460aba047c,misp_ioc 135.181.97.81,created_at 2021_02_08,updated_at 2026_06_08;)
alert ip $HOME_NET any -> 158.255.208.148 any (msg: "MISP e22 Outgoing To IP: 158.255.208.148"; flow:not_established,to_server; flowbits:isnotset,outbound_ioc; flowbits:set,o

In [13]:
# Other targets: 'snort', 'stix2', 'csv', 'text', 'netfilter', 'json' ...
csv_data = misp.search(controller='attributes', tags=['tlp:white'],
                       return_format='csv')
print(csv_data[:800])

uuid,event_id,category,type,value,comment,to_ids,date,object_relation,attribute_tag,object_uuid,object_name,object_meta_category
"8a948ce0-c2c8-4e2a-8c67-73757b48474f",22,"Other","comment","RÃ©sultats de l'investigation sur l'infrastructure d'attaque de TA505","",0,1776577177,"","DescriptionTechnique","","",""
"87ca8c58-8696-4752-8b62-34c714be0ec0",22,"Network activity","ip-dst","135.181.97.81","SDBbot C2 server [2020-11-29:]",1,1612796786,"","","","",""
"be9d342e-4921-4e85-afd4-f8201e059fb0",22,"Network activity","ip-dst","158.255.208.148","SDBbot C2 server",1,1612796786,"","","","",""
"040b18be-531c-4297-bb5b-a69595c36d96",22,"Network activity","ip-dst","158.255.208.168","SDBbot C2 server",1,1612796786,"","","","",""
"e69552e9-3236-4222-9cd3-10b0fddaa6dc",22,"Network activity","ip-dst","


---
## 5. Automate
*A small pipeline that runs itself — the takeaway artifact*

Wrap the previous segments into one reusable function. Drop it in `cron` / a scheduled task
and you have an unattended intel-to-detection feed. **This is the analyst-to-automator jump.**

In [ ]:
def export_fresh_iocs(misp, lookback='1d', outfile='misp_export.rules'):
    """Pull recent detection-flagged network IOCs and write IDS rules."""
    rules = misp.search(
        controller='attributes',
        type_attribute=['ip-dst', 'domain', 'url'],
        to_ids=True,
        date_from=lookback,
        return_format='suricata'
    )
    with open(outfile, 'w') as f:
        f.write(rules)
    print(f"Wrote rules to {outfile} (lookback={lookback})")

export_fresh_iocs(misp)

---
## Appendix — It's just REST
*Everything PyMISP did was this underneath. For any language, or full control.*

In [ ]:
import requests

headers = {'Authorization': MISP_KEY,
           'Accept': 'application/json',
           'Content-Type': 'application/json'}
body = {'returnFormat': 'json', 'type': ['ip-dst'], 'to_ids': 1, 'limit': 5}

r = requests.post(f"{MISP_URL}/attributes/restSearch",
                  headers=headers, json=body, verify=False)
attrs = r.json().get('response', {}).get('Attribute', [])
attrs[0] if attrs else "no results — fetch a feed first"


---
## 🧹 Cleanup (run between takes)
Deletes the demo event created in Section 3 so the instance resets cleanly.

In [ ]:
if created is not None:
    misp.delete_event(created.id)
    print("Deleted demo event", created.id)
    created = None
else:
    print("Nothing to clean up.")